##### 1) 라이브러리 설치

In [ ]:
%pip install -q langchain langchain-openai langchain_community tiktoken

##### 2) OpenAI 인증키 설정

In [ ]:
from dotenv import load_dotenv
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

#### CharacterTextSplitter 간단한 예제

In [ ]:
from langchain.text_splitter import CharacterTextSplitter

text = """RAG는 검색 기반의 텍스트 생성 모델입니다. 기존 언어 모델의 단점을 보완하고, 최신 정보를 제공합니다.
특히, 최신 데이터를 반영하는 데 강력한 기능을 제공합니다. 
RAG는 검색과 생성 단계를 포함합니다."""

# 마침표(".")를 기준으로 텍스트 분할
splitter = CharacterTextSplitter(chunk_size=50, chunk_overlap=10, separator=".")
chunks = splitter.split_text(text)

print(type(chunks))
print(chunks)

#### RecursiveCharacterTextSplitter 간단한 예제

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text = """RAG는 검색과 생성 단계를 포함하는 모델입니다.

이 모델은 검색 기반의 텍스트 생성 기능을 제공합니다.
특히, 최신 데이터를 반영하는 데 강력한 기능을 가지고 있습니다.

Transformer 모델을 기반으로 실시간 정보를 활용할 수 있으며, 기존의 단순한 생성 모델보다 더 정확한 답변을 제공합니다."""

# 의미 단위(문장, 단락)로 나누되, chunk_size 50 제한
splitter = RecursiveCharacterTextSplitter(chunk_size=80, chunk_overlap=20, separators=["\n\n", ".", "!", "?", " ", ""])
chunks = splitter.split_text(text)

for i, chunk in enumerate(chunks):
    print(f" Chunk {i+1}: {chunk}\n")

#### TokenTextSplitter 간단한 예제

In [ ]:
from langchain_text_splitters import TokenTextSplitter

# 파일 읽기
with open("./data/ai-terminology.txt", encoding="utf-8") as f:
    file = f.read()  # 파일 내용을 읽어오기

print("원본 텍스트 미리보기:\n", file[:500])  # 앞 500자 출력

# TokenTextSplitter 설정
text_splitter = TokenTextSplitter.from_tiktoken_encoder(
    chunk_size=200,  # 청크 크기
    chunk_overlap=20,  # 청크 간 겹치는 부분 추가하여 문맥 유지
    encoding_name="cl100k_base",  # OpenAI tiktoken 기본 인코딩 사용 (한글 처리 개선)
    add_start_index=True  # 각 청크의 시작 인덱스 반환
)

# 텍스트 분할 실행
texts = text_splitter.split_text(file)

# 결과 출력
print(f"\n🔹 총 {len(texts)}개의 청크로 분할됨.")
print("\n 첫 번째 청크:\n", texts[0])

# 청크 길이 확인
for i, chunk in enumerate(texts[:5]):  # 처음 5개만 확인
    print(f"\n🔹 Chunk {i+1} (길이: {len(chunk)}):\n{chunk}")

In [ ]:
%pip install -q transformers

In [ ]:
from transformers import GPT2TokenizerFast
from langchain.text_splitter import CharacterTextSplitter

# GPT-2 모델의 토크나이저 로드
hf_tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

# 데이터 파일 읽기
file_path = "./data/ai-terminology.txt"
with open(file_path, encoding="utf-8") as f:
    file_content = f.read()

print(" 원본 텍스트 미리보기:\n", file_content[:200])

# CharacterTextSplitter 설정 (Hugging Face 토크나이저 사용)
text_splitter = CharacterTextSplitter.from_huggingface_tokenizer(
    hf_tokenizer,
    chunk_size=300,  # 각 청크 크기 (토큰 기준 아님)
    chunk_overlap=50,  # 청크 간 중복 부분
)

# 텍스트 분할 수행
split_texts = text_splitter.split_text(file_content)

# 분할된 텍스트 출력
print(f"\n 총 {len(split_texts)}개의 청크로 분할됨\n")
for i, chunk in enumerate(split_texts[:5]):  # 처음 5개만 출력
    print(f" Chunk {i+1} ({len(chunk)}자):\n{chunk}\n")

# 토크나이저로 텍스트를 토큰 단위로 변환하여 확인
tokenized_example = hf_tokenizer.tokenize(split_texts[0])
print(f"\n 첫 번째 청크의 토큰 개수: {len(tokenized_example)}")
print(" 첫 번째 청크의 토큰 리스트:", tokenized_example[:20])  # 앞 20개만 출력
